# Part 1 - Data Exploration

This notebook reads `data/final_clean_events.csv` directly and preserves source values exactly.
Source columns are not lowercased, merged, simplified, imputed, or overwritten. Derived parsing
columns are added separately for analysis.


## Configuration and imports

Important thresholds and constants are defined in the reusable helper module. The fixed random
seed is set here even though this notebook is deterministic, so later sampling remains repeatable.


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

WORKING_DIR = Path.cwd()
REPO_ROOT = WORKING_DIR if (WORKING_DIR / "data" / "HiFPT_full_events.csv").exists() else WORKING_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

from utils.exact_event_analysis import (
    ACTION_EVENT_TYPES,
    INACTIVITY_THRESHOLDS_MINUTES,
    MISSING_SENTINEL,
    REQUIRED_COLUMNS,
    SELECTED_INACTIVITY_THRESHOLD_MINUTES,
    VIEW_EVENT_TYPES,
    add_duration_fields,
    add_exact_tokens,
    add_time_fields,
    add_token_ids,
    availability_by_event_type,
    build_session_sequences,
    build_token_dictionary,
    component_by_event_type,
    construct_clean_sessions,
    exact_tuple_counts,
    file_fingerprint,
    inferred_dtypes,
    input_csv_path,
    markdown_table,
    masked_sample,
    missing_summary,
    original_order_timestamp_issues,
    original_session_summary,
    quantile_table,
    read_events,
    representative_sequences,
    safe_json,
    save_token_outputs,
    sort_events,
    threshold_comparison,
    validate_required_columns,
    validate_tokenization,
    value_counts_with_pct,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
np.random.seed(42)

INPUT_CSV = input_csv_path(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"Input CSV: {INPUT_CSV.relative_to(REPO_ROOT)}")


Repository root: D:\BehaviourClassification
Input CSV: data\final_clean_events.csv


## Load source data and validate required columns

The CSV is loaded with string-backed source columns to avoid changing identifiers or component
labels. Numeric and datetime interpretations are derived later.


In [5]:
before_fingerprint = file_fingerprint(INPUT_CSV)
events = read_events(INPUT_CSV)
print(f"Rows after load: {len(events):,}")
print(f"Columns after load: {events.shape[1]:,}")
validate_required_columns(events)
print("Required-column validation: passed")
print("Required columns:", REQUIRED_COLUMNS)


Rows after load: 114,534
Columns after load: 15
Required-column validation: passed
Required columns: ['record_id', 'device_id', 'customer_id', 'session_id', 'event_type', 'timestamp', 'created_at', 'segment_name', 'screen_name', 'duration']


## Dataset structure

The following tables document row and column counts, inferred data types, memory usage, sample
records, duplicate identifiers, and exact component cardinalities.


In [7]:
inferred = inferred_dtypes(INPUT_CSV).rename("inferred_dtype").reset_index()
inferred.columns = ["column", "inferred_dtype"]
overview = {
    "rows": len(events),
    "columns": events.shape[1],
    "memory_usage_mb_preserved_load": round(events.memory_usage(deep=True).sum() / 1024**2, 3),
    "duplicate_record_id_rows": int(events["record_id"].duplicated(keep=False).sum()),
    "duplicate_record_id_excess": int(events["record_id"].duplicated().sum()),
    "fully_duplicated_rows": int(events.drop(columns=["source_row_number"]).duplicated().sum()),
    "devices": int(events["device_id"].nunique(dropna=True)),
    "customers": int(events["customer_id"].nunique(dropna=True)),
    "original_sessions": int(events["session_id"].nunique(dropna=True)),
    "event_types": int(events["event_type"].nunique(dropna=True)),
    "distinct_segment_names": int(events["segment_name"].nunique(dropna=True)),
    "distinct_screen_names": int(events["screen_name"].nunique(dropna=True)),
    "exact_event_tuples": int(events.drop_duplicates(["event_type", "segment_name", "screen_name"]).shape[0]),
}
print(safe_json(overview))
print("\nColumn names:")
print(events.drop(columns=["source_row_number"]).columns.tolist())
print("\nInferred dtypes when pandas reads the CSV normally:")
print(markdown_table(inferred, max_rows=50))
print("\nMasked sample:")
print(markdown_table(masked_sample(events.drop(columns=["source_row_number"]), n=8), max_rows=8))


{
  "rows": 114534,
  "columns": 15,
  "memory_usage_mb_preserved_load": 98.279,
  "duplicate_record_id_rows": 0,
  "duplicate_record_id_excess": 0,
  "fully_duplicated_rows": 0,
  "devices": 182,
  "customers": 154,
  "original_sessions": 1425,
  "event_types": 2,
  "distinct_segment_names": 1617,
  "distinct_screen_names": 611,
  "exact_event_tuples": 3338
}

Column names:
['record_id', 'utm_source', 'device_id', 'customer_id', 'session_id', 'created_at', 'timestamp', 'event_type', 'OS', 'segment_name', 'visit', 'duration', 'screen_name', 'segmentation.external_id']

Inferred dtypes when pandas reads the CSV normally:
| column | inferred_dtype |
| --- | --- |
| record_id | str |
| utm_source | str |
| device_id | str |
| customer_id | float64 |
| session_id | str |
| created_at | str |
| timestamp | int64 |
| event_type | str |
| OS | str |
| segment_name | str |
| visit | float64 |
| duration | int64 |
| screen_name | str |
| segmentation.external_id | str |

Masked sample:
| record

## Missing-value exploration

Missing values are counted exactly as loaded. Some missingness may be structural, but this notebook
does not assign business meaning to it without source-owner confirmation.


In [9]:
miss = missing_summary(events.drop(columns=["source_row_number"]))
by_event_availability = availability_by_event_type(events)
both_missing = int(events["segment_name"].isna().mul(events["screen_name"].isna()).sum())
segment_only = int(events["segment_name"].notna().mul(events["screen_name"].isna()).sum())
screen_only = int(events["segment_name"].isna().mul(events["screen_name"].notna()).sum())
print("Rows before missing-value analysis:", len(events))
print("\nMissing summary:")
miss_display = miss.reset_index()
miss_display.columns = ["column", "missing_count", "missing_percentage"]
print(markdown_table(miss_display, max_rows=50))
print("\nAvailability by event_type:")
print(markdown_table(by_event_availability, max_rows=50))
print("\nSegment/screen missingness combinations:")
print(safe_json({
    "both_segment_name_and_screen_name_missing": both_missing,
    "segment_present_screen_missing": segment_only,
    "segment_missing_screen_present": screen_only,
}))


Rows before missing-value analysis: 114534

Missing summary:
| column | missing_count | missing_percentage |
| --- | --- | --- |
| utm_source | 112405 | 98.1412 |
| segmentation.external_id | 110063 | 96.0964 |
| screen_name | 87135 | 76.0778 |
| visit | 71678 | 62.5823 |
| customer_id | 16681 | 14.5642 |
| segment_name | 23 | 0.0201 |
| record_id | 0 | 0.0 |
| device_id | 0 | 0.0 |
| session_id | 0 | 0.0 |
| created_at | 0 | 0.0 |
| timestamp | 0 | 0.0 |
| event_type | 0 | 0.0 |
| OS | 0 | 0.0 |
| duration | 0 | 0.0 |

Availability by event_type:
| event_type | rows | segment_name_available | segment_name_missing | segment_name_available_pct | screen_name_available | screen_name_missing | screen_name_available_pct | duration_available | duration_missing | duration_available_pct |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Action | 27844 | 27821 | 23 | 99.9174 | 27399 | 445 | 98.4018 | 27844 | 0 | 100.0 |
| View | 86690 | 86690 | 0 | 100.0 | 0 | 86690 | 0.0 |

## Event and component distributions

Values are counted exactly. No platform prefixes, suffixes, paths, URLs, spellings, or semantic
variants are merged.


In [11]:
event_counts = value_counts_with_pct(events["event_type"])
top_segments = value_counts_with_pct(events["segment_name"], top_n=25)
top_screens = value_counts_with_pct(events["screen_name"], top_n=25)
top_segment_by_event = component_by_event_type(events, "segment_name", top_n=10)
top_screen_by_event = component_by_event_type(events, "screen_name", top_n=10)
exact_counts = exact_tuple_counts(events)
singleton_count = int(exact_counts["event_frequency"].eq(1).sum())
singleton_pct = singleton_count / len(exact_counts) * 100
long_tail = {
    "exact_token_vocabulary_size": len(exact_counts),
    "singleton_exact_token_combinations": singleton_count,
    "singleton_percentage_of_vocabulary": round(singleton_pct, 4),
    "top_10_combinations_event_share_pct": round(exact_counts.head(10)["event_frequency"].sum() / len(events) * 100, 4),
    "top_100_combinations_event_share_pct": round(exact_counts.head(100)["event_frequency"].sum() / len(events) * 100, 4),
}
print("Rows before distribution analysis:", len(events))
print("\nEvent-type counts:")
print(markdown_table(event_counts, max_rows=25))
print("\nTop segment names overall:")
print(markdown_table(top_segments, max_rows=25))
print("\nTop segment names by event_type:")
print(markdown_table(top_segment_by_event, max_rows=30))
print("\nTop screen names overall:")
print(markdown_table(top_screens, max_rows=25))
print("\nTop screen names by event_type:")
print(markdown_table(top_screen_by_event, max_rows=30))
print("\nMost frequent exact tuples:")
print(markdown_table(exact_counts.head(25), max_rows=25))
print("\nExact-token frequency distribution and long-tail summary:")
print(safe_json(long_tail))
print("\nFrequency quantiles:")
print(markdown_table(quantile_table(exact_counts["event_frequency"], [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))


Rows before distribution analysis: 114534

Event-type counts:
| event_type | count | percentage |
| --- | --- | --- |
| View | 86690 | 75.6893 |
| Action | 27844 | 24.3107 |

Top segment names overall:
| segment_name | count | percentage |
| --- | --- | --- |
| MainTabBarController | 7049 | 6.1545 |
| WebkitEcommerceController | 5643 | 4.9269 |
| HomeVC | 5487 | 4.7907 |
| BaseNavigation | 3764 | 3.2864 |
| HOME | 3124 | 2.7276 |
| android/Home | 3116 | 2.7206 |
| UITrackingElementWindowController | 2638 | 2.3032 |
| btn_back | 2610 | 2.2788 |
| UIViewController | 2145 | 1.8728 |
| SplashVC | 1974 | 1.7235 |
| ServiceManageVC | 1834 | 1.6013 |
| UIHostingController<PopupView> | 1585 | 1.3839 |
| LoginVC | 1533 | 1.3385 |
| HiWebViewActivity | 1473 | 1.2861 |
| ChooseContractVC | 1426 | 1.245 |
| AccountHomeController | 1379 | 1.204 |
| android/home/home_service_management | 1260 | 1.1001 |
| android/Home/Contract_List | 1239 | 1.0818 |
| _UISceneHostingViewController | 1188 | 1.0372 |


## Timestamp analysis

`timestamp` and `created_at` are parsed into derived columns. The epoch unit for `timestamp` is
selected from evidence by comparing candidate units against `created_at`.


In [13]:
events_with_time, time_metadata = add_time_fields(events)
print(f"Rows after timestamp parsing: {len(events_with_time):,}")
print("\nTimestamp unit evidence:")
print(markdown_table(time_metadata["timestamp_unit_evidence"], max_rows=10))
print("\nPrimary ordering decision:")
print(safe_json({
    "selected_timestamp_unit": time_metadata["selected_timestamp_unit"],
    "primary_event_time_column": time_metadata["primary_event_time_column"],
    "secondary_event_time_column": time_metadata["secondary_event_time_column"],
    "reason": time_metadata["primary_event_time_reason"],
    "deterministic_order": ["session_id", "primary_event_time", "secondary_event_time", "record_id", "source_row_number"],
}))
timestamp_summary = {
    "timestamp_datetime_min": str(events_with_time["timestamp_datetime"].min()),
    "timestamp_datetime_max": str(events_with_time["timestamp_datetime"].max()),
    "created_at_datetime_min": str(events_with_time["created_at_datetime"].min()),
    "created_at_datetime_max": str(events_with_time["created_at_datetime"].max()),
    "missing_timestamp_count": int(events_with_time["timestamp"].isna().sum()),
    "invalid_timestamp_count": int(events_with_time["timestamp_datetime"].isna().sum() - events_with_time["timestamp"].isna().sum()),
    "missing_created_at_count": int(events_with_time["created_at"].isna().sum()),
    "invalid_created_at_count": int(events_with_time["created_at_datetime"].isna().sum() - events_with_time["created_at"].isna().sum()),
}
print("\nTimestamp summary:")
print(safe_json(timestamp_summary))
disagreement = events_with_time["timestamp_disagreement_seconds"].dropna()
large_threshold = disagreement.quantile(.99) if len(disagreement) else np.nan
large_disagreement = events_with_time.loc[
    events_with_time["timestamp_disagreement_seconds"].gt(large_threshold),
    ["record_id", "session_id", "timestamp", "created_at", "timestamp_datetime", "created_at_datetime", "timestamp_disagreement_seconds"],
].sort_values("timestamp_disagreement_seconds", ascending=False).head(10)
print("\nTimestamp disagreement quantiles:")
print(markdown_table(quantile_table(disagreement, [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))
print("\nEvents with unusually large timestamp disagreement:")
print(markdown_table(large_disagreement, max_rows=10))


Rows after timestamp parsing: 114,534

Timestamp unit evidence:
| candidate_unit | valid_count | min_parsed | max_parsed | median_abs_diff_vs_created_at_seconds |
| --- | --- | --- | --- | --- |
| s | 114534 | 1970-10-28 14:15:23+00:00 | 1970-09-05 16:08:16+00:00 | 1781202441565.1597 |
| ms | 114534 | 2026-07-01 00:59:53.723000+00:00 | 2026-07-07 04:44:38.896000+00:00 | 1.03 |
| us | 114534 | 1970-01-21 15:14:27.593723+00:00 | 1970-01-21 15:23:19.478896+00:00 | 1781202454.1583545 |
| ns | 114534 | 1970-01-01 00:29:42.867593723+00:00 | 1970-01-01 00:29:43.399478896+00:00 | 1782983656.6115613 |

Primary ordering decision:
{
  "selected_timestamp_unit": "ms",
  "primary_event_time_column": "timestamp_datetime",
  "secondary_event_time_column": "created_at_datetime",
  "reason": "timestamp has at least as many valid values and at least as many distinct event times as created_at",
  "deterministic_order": [
    "session_id",
    "primary_event_time",
    "secondary_event_time",
    "record_

## Duration analysis

The source `duration` column is preserved. Numeric, log, missingness, zero, negative, positive, and
outlier flags are derived separately. Duration is not included in categorical event tokens.


In [15]:
events_with_duration = add_duration_fields(events_with_time)
print(f"Rows after duration derivation: {len(events_with_duration):,}")
duration_numeric = events_with_duration["duration_numeric"]
duration_stats = {
    "missing_duration_count": int(duration_numeric.isna().sum()),
    "zero_duration_count": int(duration_numeric.eq(0).sum()),
    "negative_duration_count": int(duration_numeric.lt(0).sum()),
    "positive_duration_count": int(duration_numeric.gt(0).sum()),
    "minimum": float(duration_numeric.min()) if duration_numeric.notna().any() else None,
    "maximum": float(duration_numeric.max()) if duration_numeric.notna().any() else None,
    "mean": float(duration_numeric.mean()) if duration_numeric.notna().any() else None,
    "median": float(duration_numeric.median()) if duration_numeric.notna().any() else None,
}
duration_by_event = events_with_duration.groupby("event_type", dropna=False).agg(
    rows=("record_id", "size"),
    duration_available=("duration_numeric", lambda s: int(s.notna().sum())),
    duration_missing=("duration_numeric", lambda s: int(s.isna().sum())),
    duration_zero=("duration_zero", "sum"),
    duration_positive=("duration_positive", "sum"),
).reset_index()
extremes = events_with_duration.sort_values("duration_numeric", ascending=False).head(10)[
    ["record_id", "session_id", "event_type", "segment_name", "screen_name", "duration", "duration_numeric"]
]
print("\nDuration summary:")
print(safe_json(duration_stats))
print("\nDuration percentiles:")
print(markdown_table(quantile_table(duration_numeric, [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))
print("\nDuration availability by event_type:")
print(markdown_table(duration_by_event, max_rows=20))
print("\nExtreme-duration records:")
print(markdown_table(extremes, max_rows=10))
print("\nlog1p(duration) percentiles for non-negative durations:")
print(markdown_table(quantile_table(events_with_duration["log1p_duration"], [0, .25, .5, .75, .9, .95, .99, 1]), max_rows=20))


Rows after duration derivation: 114,534

Duration summary:
{
  "missing_duration_count": 0,
  "zero_duration_count": 74949,
  "negative_duration_count": 0,
  "positive_duration_count": 39585,
  "minimum": 0.0,
  "maximum": 98982.0,
  "mean": 27.24852882113608,
  "median": 0.0
}

Duration percentiles:
| quantile | value |
| --- | --- |
| 0.0 | 0.0 |
| 0.25 | 0.0 |
| 0.5 | 0.0 |
| 0.75 | 2.0 |
| 0.9 | 16.0 |
| 0.95 | 47.0 |
| 0.99 | 262.0 |
| 1.0 | 98982.0 |

Duration availability by event_type:
| event_type | rows | duration_available | duration_missing | duration_zero | duration_positive |
| --- | --- | --- | --- | --- | --- |
| Action | 27844 | 27844 | 0 | 27844 | 0 |
| View | 86690 | 86690 | 0 | 47105 | 39585 |

Extreme-duration records:
| record_id | session_id | event_type | segment_name | screen_name | duration | duration_numeric |
| --- | --- | --- | --- | --- | --- | --- |
| 6a448c8509279a6131b59737 | 71B0F0B9-CA6B-4CF3-A1F2-69CEEA4BBEB1 | View | MainTabBarController | <MISSING>

## Part 1 summary

This notebook stops at exploration. It does not perform semantic canonicalization or journey
algorithm work.


In [17]:
after_fingerprint = file_fingerprint(INPUT_CSV)
assert before_fingerprint == after_fingerprint, "Input CSV changed during exploration."
print("Input file unchanged: passed")
print(safe_json({
    "dataset_size": len(events),
    "columns": events.shape[1],
    "event_types_observed": events["event_type"].dropna().unique().tolist(),
    "exact_tuple_count": int(events.drop_duplicates(["event_type", "segment_name", "screen_name"]).shape[0]),
    "input_file_validation": "unchanged",
}))


Input file unchanged: passed
{
  "dataset_size": 114534,
  "columns": 15,
  "event_types_observed": [
    "View",
    "Action"
  ],
  "exact_tuple_count": 3338,
  "input_file_validation": "unchanged"
}
